## Calling LLM

In [7]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-3.5-turbo",
        input=prompt
    )
    return response.output_text

In [3]:
llm("Hey, what's up?")

"Hello! I'm here to help you with any questions or tasks you have. How can I assist you today?"

In [4]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm happy to hear that you're interested in joining the course! You may need to check with the course instructor or the institution offering the course to see if there are specific enrollment deadlines or requirements. Typically, online courses have flexible enrollment options, but it's always best to confirm with the course provider to ensure a smooth enrollment process.


In [5]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [6]:
answer = llm(prompt)
print(answer)

If you just discovered the course, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. 

As for the confirmation email for the LLM Zoomcamp, you don't need one. You are already accepted, and you can start learning and submitting homework, provided the submission form is open. Registration is more about gauging interest and is not checked against a registered list.

Regarding the video/Zoom link for "Office Hours" or live/workshop sessions, the Zoom link is only shared with instructors, presenters, and TAs. Students usually participate via YouTube Live and submit questions using Slido.

For cloud alternatives with GPU, it's important to carefully check the quota and reset cycle. Some potential options include Google Colab, Kaggle, and Databricks.


## RAG Search

In [8]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [11]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [13]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [14]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

## RAG Pipeline

In [21]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from rag_base.rag_helper import RAGBase
from rag_base.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

QUESTION: Can I join the course now and still receive a certificate?

ANSWER: Yes, you can still join the course now, but to receive a certificate, you need to submit your project while submissions are still being accepted. Remember that certificates are awarded only if you finish the course with a "live" cohort and complete the required peer reviews.


In [22]:
answer_2 = assistant.rag("How do I get a certificate?")
print(answer_2)

I don't know.


In [23]:
answer_3 = assistant.rag("Can I still join the course after it started?")
print(answer_3)

Yes, you can join the course after it has started. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.
